# SQL Basics with DuckDB & Pandas DataFrames

This notebook introduces the most common SQL operations and shows the **equivalent Pandas DataFrame code** side by side, using a small Quran-themed dataset:

- `data/quran_surahs.csv` — metadata for all 114 surahs (chapter number, names, revelation place, ayah count)
- `data/quran_ayahs.csv` — a sample of ayahs (verses) from the last 12 short surahs (Juz 30), with simplified paraphrased text for demo purposes only (not a scholarly translation)

For every concept below you'll see:
1. A short explanation
2. The **DuckDB SQL** way
3. The **Pandas DataFrame** way

The goal is to build intuition for how SQL clauses map to DataFrame methods.


In [4]:
# Install duckdb + jupysql if not already available (uncomment if needed)
# %pip install duckdb duckdb-engine jupysql

import duckdb
import pandas as pd

# Load the CSV data into Pandas DataFrames
surahs = pd.read_csv("data/quran_surahs.csv")
ayahs = pd.read_csv("data/quran_ayahs.csv")

# Register the DataFrames as DuckDB views so %%sql cells can query them by name
conn = duckdb.connect()
conn.register("surahs", surahs)
conn.register("ayahs", ayahs)

# jupysql lets us write real SQL cells (%%sql) instead of embedding SQL in Python strings
%load_ext sql
%sql conn --alias duckdb
%config SqlMagic.displaycon = False
%config SqlMagic.feedback = False

print(surahs.shape, ayahs.shape)
surahs.head()


The sql extension is already loaded. To reload it, use:
  %reload_ext sql
(114, 5) (60, 4)


,surah_no,surah_name_en,surah_name,revelation_place,total_ayahs
0,1,The Opening,الفاتحة,Meccan,7
1,2,The Cow,البقرة,Medinan,286
2,3,The Family of Imran,آل عمران,Medinan,200
3,4,The Women,النساء,Medinan,176
4,5,The Table Spread,المائدة,Medinan,120


> **Tip:** Because DuckDB can see Pandas DataFrames by variable name, `%%sql` cells below can `SELECT ... FROM surahs` directly — no need to load the data into a database first or wrap SQL in a Python string.

## 1. Selecting Columns

**SQL:** the `SELECT` clause picks which columns to return.
**Pandas:** double square brackets `df[[...]]` picks a list of columns.


In [5]:
%%sql
-- SQL: select specific columns
SELECT surah_no, surah_name_en, surah_name, total_ayahs
FROM surahs
LIMIT 5


surah_no,surah_name_en,surah_name,total_ayahs
1,The Opening,الفاتحة,7
2,The Cow,البقرة,286
3,The Family of Imran,آل عمران,200
4,The Women,النساء,176
5,The Table Spread,المائدة,120


In [6]:
# Pandas: same result using column selection
surahs[["surah_no", "surah_name_en", "surah_name", "total_ayahs"]].head()


,surah_no,surah_name_en,surah_name,total_ayahs
0,1,The Opening,الفاتحة,7
1,2,The Cow,البقرة,286
2,3,The Family of Imran,آل عمران,200
3,4,The Women,النساء,176
4,5,The Table Spread,المائدة,120


## 2. Filtering Rows

**SQL:** the `WHERE` clause keeps rows matching a condition.
**Pandas:** boolean indexing `df[condition]` does the same thing.

Example: find all **Medinan** surahs.


In [7]:
%%sql
-- SQL: filter with WHERE
SELECT surah_no, surah_name_en, revelation_place
FROM surahs
WHERE revelation_place = 'Medinan'
LIMIT 5


surah_no,surah_name_en,revelation_place
2,The Cow,Medinan
3,The Family of Imran,Medinan
4,The Women,Medinan
5,The Table Spread,Medinan
8,The Spoils of War,Medinan


In [8]:
# Pandas: filter with a boolean mask
mask = surahs["revelation_place"] == "Medinan"
surahs.loc[mask, ["surah_no", "surah_name_en", "revelation_place"]].head()


,surah_no,surah_name_en,revelation_place
1,2,The Cow,Medinan
2,3,The Family of Imran,Medinan
3,4,The Women,Medinan
4,5,The Table Spread,Medinan
7,8,The Spoils of War,Medinan


## 3. Sorting Rows

**SQL:** `ORDER BY column [ASC|DESC]`.
**Pandas:** `df.sort_values(by=column, ascending=True/False)`.

Example: the 5 shortest surahs by ayah count.


In [9]:
%%sql
-- SQL: sort with ORDER BY, keep top 5 with LIMIT
SELECT surah_no, surah_name_en, total_ayahs
FROM surahs
ORDER BY total_ayahs ASC
LIMIT 5


surah_no,surah_name_en,total_ayahs
103,The Declining Day,3
108,The Abundance,3
110,The Divine Support,3
106,Quraysh,4
112,The Sincerity,4


In [10]:
# Pandas: sort_values + head() as the equivalent of ORDER BY + LIMIT
surahs.sort_values(by="total_ayahs", ascending=True)[["surah_no", "surah_name_en", "total_ayahs"]].head(5)


,surah_no,surah_name_en,total_ayahs
109,110,The Divine Support,3
102,103,The Declining Day,3
107,108,The Abundance,3
111,112,The Sincerity,4
105,106,Quraysh,4


## 4. Joining Tables

**SQL:** `JOIN ... ON` combines rows from two tables that share a key.
**Pandas:** `df.merge(other_df, on=key, how=...)` does the same thing.

Here we join the `ayahs` table (verses) with the `surahs` table (chapter metadata) on `surah_no`, so each verse shows its surah's name.


In [11]:
%%sql
-- SQL: INNER JOIN ayahs to surahs on surah_no
SELECT s.surah_name_en, s.surah_name, a.ayah_no, a.ayah_text
FROM ayahs AS a
JOIN surahs AS s ON a.surah_no = s.surah_no
WHERE a.surah_no = 112


surah_name_en,surah_name,ayah_no,ayah_text
The Sincerity,الإخلاص,4,ولم يكن له كفوا أحد
The Sincerity,الإخلاص,3,لم يلد ولم يولد
The Sincerity,الإخلاص,2,الله الصمد
The Sincerity,الإخلاص,1,قل هو الله أحد


In [12]:
# Pandas: merge() is the equivalent of JOIN
joined = ayahs.merge(surahs, on="surah_no", how="inner")
joined.loc[joined["surah_no"] == 112, ["surah_name_en", "ayah_no", "ayah_text"]]


,surah_name_en,ayah_no,ayah_text
45,The Sincerity,1,قل هو الله أحد
46,The Sincerity,2,الله الصمد
47,The Sincerity,3,لم يلد ولم يولد
48,The Sincerity,4,ولم يكن له كفوا أحد


## 5. Grouping & Aggregation

**SQL:** `GROUP BY column` + aggregate functions (`COUNT`, `SUM`, `AVG`, ...).
**Pandas:** `df.groupby(column).agg(...)` does the same thing.

Example: count how many surahs are Meccan vs Medinan, and the average number of ayahs in each group.


In [13]:
%%sql
-- SQL: GROUP BY with aggregate functions
SELECT
    revelation_place,
    COUNT(*) AS surah_count,
    AVG(total_ayahs) AS avg_ayahs
FROM surahs
GROUP BY revelation_place


revelation_place,surah_count,avg_ayahs
Medinan,28,57.964285714285715
Meccan,86,53.63953488372093


In [14]:
# Pandas: groupby() + agg() is the equivalent of GROUP BY
surahs.groupby("revelation_place").agg(
    surah_count=("surah_no", "count"),
    avg_ayahs=("total_ayahs", "mean")
).reset_index()


,revelation_place,surah_count,avg_ayahs
0,Meccan,86,53.639535
1,Medinan,28,57.964286


## 6. Other Essential Operations

A few more common building blocks:

- **DISTINCT values:** `SELECT DISTINCT col` ↔ `df["col"].unique()` / `df.drop_duplicates(subset=...)`
- **Computed column:** `SELECT col1 + col2 AS new_col` ↔ `df["new_col"] = df["col1"] + df["col2"]`
- **Filtering on an aggregate (`HAVING`):** `GROUP BY ... HAVING COUNT(*) > n` ↔ filter the DataFrame *after* `groupby().agg()`

Example: get the distinct revelation places, then find surahs whose ayah count is above the overall average (a computed comparison).


In [15]:
%%sql
-- SQL: DISTINCT values
SELECT DISTINCT revelation_place FROM surahs


revelation_place
Medinan
Meccan


In [16]:
# Pandas: unique() is the equivalent of DISTINCT on one column
surahs["revelation_place"].unique()


<StringArray>
['Meccan', 'Medinan']
Length: 2, dtype: str

In [17]:
%%sql
-- SQL: computed column (WITH clause holds the average) + filter above it (like HAVING, but on rows here)
WITH stats AS (
    SELECT AVG(total_ayahs) AS avg_ayahs FROM surahs
)
SELECT s.surah_no, s.surah_name_en, s.total_ayahs,
       s.total_ayahs - stats.avg_ayahs AS diff_from_avg
FROM surahs AS s, stats
WHERE s.total_ayahs > stats.avg_ayahs
ORDER BY diff_from_avg DESC
LIMIT 5


surah_no,surah_name_en,total_ayahs,diff_from_avg
2,The Cow,286,231.2982456140351
26,The Poets,227,172.2982456140351
7,The Heights,206,151.2982456140351
3,The Family of Imran,200,145.2982456140351
37,Those Ranged in Ranks,182,127.2982456140351


In [18]:
# Pandas: compute a new column, then filter rows above the average
avg_ayahs = surahs["total_ayahs"].mean()
surahs["diff_from_avg"] = surahs["total_ayahs"] - avg_ayahs

result = surahs[surahs["total_ayahs"] > avg_ayahs].sort_values("diff_from_avg", ascending=False)
result[["surah_no", "surah_name_en", "total_ayahs", "diff_from_avg"]].head(5)


,surah_no,surah_name_en,total_ayahs,diff_from_avg
1,2,The Cow,286,231.298246
25,26,The Poets,227,172.298246
6,7,The Heights,206,151.298246
2,3,The Family of Imran,200,145.298246
36,37,Those Ranged in Ranks,182,127.298246


## Cheat Sheet: SQL ↔ Pandas

| Concept | SQL | Pandas |
|---|---|---|
| Select columns | `SELECT col1, col2 FROM t` | `df[["col1", "col2"]]` |
| Filter rows | `WHERE condition` | `df[condition]` |
| Sort | `ORDER BY col ASC/DESC` | `df.sort_values("col", ascending=True/False)` |
| Limit rows | `LIMIT n` | `df.head(n)` |
| Join | `JOIN ... ON key` | `df.merge(other, on="key", how=...)` |
| Group + aggregate | `GROUP BY col` + `COUNT/AVG/SUM` | `df.groupby("col").agg(...)` |
| Distinct values | `SELECT DISTINCT col` | `df["col"].unique()` |
| Computed column | `SELECT col1 + col2 AS new_col` | `df["new_col"] = df["col1"] + df["col2"]` |
| Filter on aggregate | `GROUP BY ... HAVING ...` | filter after `groupby().agg()` |

Both DuckDB SQL and Pandas express the same relational logic — SQL is declarative (describe *what* you want), while Pandas is a chain of explicit steps (describe *how* to get it).
